Adopted from `preprocess_raw_support2.R` and `find_covariate_order.R` in the T-Vine-Synth repository by Griesbauer

In [36]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

In [37]:
# Fetch dataset 
support2 = fetch_ucirepo(id=880) 
df = support2.data.original

In [38]:
# Pick subset of columns
cols_exclude = ["id", "sex", "dzgroup", "dzclass", "edu", "income", "avtisst", "race", 
                "prg6m", "dnr", "glucose", "glucose", "urine", "adlp", "sfdm2",
                "d.time", "hospdead", "ca", "dementia", "diabetes", "adls", "adlsc"]
df.drop(labels=cols_exclude, axis=1, inplace=True)

In [39]:
# Drop missing values
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

In [40]:
# Reorder columns and define response variable
df.rename({"death": "Y"}, axis=1, inplace=True)
response = ["Y"]
col_reorder = np.append(np.setdiff1d(df.columns, response), response)
df = df[col_reorder]

In [41]:
# Train-test-split
df.index.name = "id"
n, d = df.shape
n_train = int(np.ceil(n*0.8))
rng = np.random.default_rng(1412)
train_ind = rng.choice(np.arange(n), n_train, replace=False)
test_ind = np.setdiff1d(np.arange(n), train_ind)
df_train = df.iloc[train_ind]
df_test = df.iloc[test_ind]

In [42]:
# Order columns using Algorithm 1 of Griesbauer et al. (2025)
def algo1(real_data: pd.DataFrame, 
          sa: list[str], 
          response: str, 
          rho: float) -> list[str]:
    """
    Order columns in dataset according to Algorithm 1 in TVineSynth (Griesbauer et al., 2025).

    Args:
        real_data (pd.DataFrame): Input dataframe.
        sa (list[str]): Names of sensitive attributes.
        response (str): Name of the response column.
        rho (float): Threshold on absolute pairwise association, a value between 0 and 1.
        
    Returns:
        New order of columns in input dataframe.
    """
    d = real_data.shape[1]
    attributes = np.setdiff1d(real_data.columns, response)
    non_sa = np.setdiff1d(attributes, sa)
    
    # Compute pairwise associations with sensitive attributes
    pa = real_data.corr(method="kendall").abs().loc[sa, non_sa]
    
    # Identify attributes with strong pairwise association with sensitive attributes
    K = non_sa[(pa > rho).any(axis=0)]
    K_sorted = K[np.argsort(-pa.loc[:,K].max().values)]
    
    # Define new order
    O_star = np.concatenate([sa, 
                             K_sorted, 
                             np.setdiff1d(non_sa, K_sorted),
                             [response]])

    return(O_star)



# Reorder columns
sa = ["totcst", "crea"] # sensitive attributes
rho = 0.6
response = df.columns[-1]
O_star = algo1(df_train, sa, response, rho)
df_train = df_train[O_star]
df_test = df_test[O_star]


In [43]:
# Save data sets with ordered columns
df_train.to_csv("../data/real_support2_small.csv", index=False)
df_test.to_csv("../data/test_support2_small.csv", index=False)

In [44]:
# Create to json files
import json

def df_to_schema(df, categorical_cols):
    columns = []
    for col in df.columns:
        if col in categorical_cols:
            categories = df[col].astype(str).unique().tolist()
            columns.append({"name": col,
                            "type": "Categorical",
                            "size": len(categories),
                            "i2s": categories})
        else:
            columns.append({"name": col,
                            "type": "Float",
                            "min": float(df[col].min()),
                            "max": float(df[col].max())})
    return {"columns": columns}

# Save json files
with open("../data/real_support2_small.json", "w") as file:
    json.dump(df_to_schema(df_train, categorical_cols=["Y"]), file, indent=2)

with open("../data/test_support2_small.json", "w") as file:
    json.dump(df_to_schema(df_test, categorical_cols=["Y"]), file, indent=2)